# 11. HNSW — 왜 계층이 필요한가

슬라이드 「HNSW 원리 3·4」.
- 새 노드는 **무작위로 최고 층**을 받는다 (지수분포 → 위층일수록 노드가 기하급수로 적다)
- 검색은 **맨 위층에서 그리디로 멀리 점프** → 한 층씩 내려가며 좁힘 → 0층에서 `ef` 후보로 넓게 확인

논문(Malkov & Yashunin, 2016)의 알고리즘을 numpy 로 100줄 남짓에 구현한다.
pgvector·FAISS 의 HNSW 도 뼈대는 같다.

| 셀 | 하는 일 |
|---|---|
| 1 | HNSW 구현 (논문 알고리즘) |
| 2 | 층 배정 — 100만 개의 지수분포 |
| 3 | 2차원에서 층별 모습과 검색 경로 |
| 4 | 계층의 효과 — HNSW vs 평면 NSW |
| 5 | N 이 16배가 될 때 비교 횟수 |
| 6 | ef_search 와 이웃 선택 방식 |

**Run → Run All Cells** 로 한 번에 실행하거나, 셀을 하나씩 **Shift + Enter**

### 셀 1 · HNSW 구현 (논문 알고리즘)

In [ ]:
# 셀 1 · HNSW 구현 (논문 알고리즘)
from labutil import *
import heapq


class HNSW:
    def __init__(self, M=8, ef_construction=64, heuristic=True, seed=0):
        self.M, self.M0 = M, 2 * M            # 위층 이웃 수 M, 0층은 2M (논문 권장)
        self.efc = ef_construction
        self.mL = 1 / np.log(M)               # 층 배정 계수: P(층 ≥ l) = M^(-l)
        self.heuristic = heuristic
        self.rng = np.random.default_rng(seed)
        self.links, self.entry, self.top = [], None, -1
        self.n_dist = 0                       # 거리 계산 횟수 (비용 측정용)

    def _dist(self, q, ids):
        self.n_dist += len(ids)
        return np.linalg.norm(self.X[ids] - q, axis=1)

    def _search_layer(self, q, eps, ef, lev, trace=None):
        """한 층 안에서 후보 ef 개를 들고 다니는 탐색 (10번의 beam 과 같다)."""
        d = self._dist(q, eps)
        visited = set(eps)
        cand = [(di, e) for di, e in zip(d, eps)]; heapq.heapify(cand)
        best = [(-di, e) for di, e in zip(d, eps)]; heapq.heapify(best)
        while len(best) > ef:
            heapq.heappop(best)
        while cand:
            dc, c = heapq.heappop(cand)
            if dc > -best[0][0]:
                break
            if trace is not None:
                trace.append((lev, c))
            new = [n for n in self.links[c][lev] if n not in visited]
            if not new:
                continue
            visited.update(new)
            for dn, n in zip(self._dist(q, new), new):
                if len(best) < ef or dn < -best[0][0]:
                    heapq.heappush(cand, (dn, n)); heapq.heappush(best, (-dn, n))
                    if len(best) > ef:
                        heapq.heappop(best)
        return sorted((-dn, n) for dn, n in best)          # [(거리, id)] 가까운 순

    def _select(self, base, cands, m):
        """이웃 고르기. cands = [(거리, id)] 가까운 순."""
        if not self.heuristic:
            return [n for _, n in cands[:m]]                # 단순: 가장 가까운 m 개
        out = []                                            # 휴리스틱: 이미 고른 이웃보다
        for dc, c in cands:                                 # base 에 더 가까운 후보만 채택
            if len(out) == m:
                break
            if not out or dc < self._dist(self.X[c], out).min():
                out.append(c)
        return out

    def build(self, X):
        self.X = np.asarray(X, dtype=np.float32)
        for i in range(len(self.X)):
            self._insert(i)
        return self

    def _insert(self, i):
        q = self.X[i]
        lvl = int(-np.log(self.rng.random()) * self.mL)    # 이 노드의 최고 층
        self.links.append([[] for _ in range(lvl + 1)])
        if self.entry is None:
            self.entry, self.top = i, lvl
            return
        ep = [self.entry]
        for lev in range(self.top, lvl, -1):                # 내 층보다 위에서는 그리디로 내려오기만
            ep = [self._search_layer(q, ep, 1, lev)[0][1]]
        for lev in range(min(lvl, self.top), -1, -1):       # 내 층부터 0층까지 이웃 연결
            W = self._search_layer(q, ep, self.efc, lev)
            m_max = self.M0 if lev == 0 else self.M
            for n in self._select(q, W, self.M):
                self.links[i][lev].append(n)
                self.links[n][lev].append(i)
                if len(self.links[n][lev]) > m_max:         # 넘치면 n 의 이웃을 다시 고른다
                    nb = self.links[n][lev]
                    dd = self._dist(self.X[n], nb)
                    self.links[n][lev] = self._select(self.X[n], sorted(zip(dd, nb)), m_max)
            ep = [n for _, n in W]
        if lvl > self.top:
            self.entry, self.top = i, lvl

    def search(self, q, k=10, ef=32, trace=None, flat=False):
        """flat=True 면 위층을 건너뛰고 0층에서 바로 시작 (= 계층 없는 NSW)."""
        q = np.asarray(q, dtype=np.float32)
        ep = [self.entry]
        if not flat:
            for lev in range(self.top, 0, -1):
                ep = [self._search_layer(q, ep, 1, lev, trace)[0][1]]
        return [n for _, n in self._search_layer(q, ep, max(ef, k), 0, trace)[:k]]

## 1. 층 배정 — 지수분포
`층 = floor(−ln(U) × mL)`, `mL = 1/ln(M)` → 한 층 올라갈 때마다 노드 수가 1/M 로 준다

### 셀 2 · 층 배정 — 100만 개의 지수분포

In [ ]:
# 셀 2 · 층 배정 — 100만 개의 지수분포
M, N = 16, 1_000_000
rng = np.random.default_rng(0)
levels = (-np.log(rng.random(N)) / np.log(M)).astype(int)
cnt = pd.Series(levels).value_counts().sort_index()
pd.DataFrame({"이 층까지 올라간 노드": cnt[::-1].cumsum()[::-1],
              "기대값 N·M^(-층)": [round(N * M ** -l) for l in cnt.index]}).rename_axis("층")

100만 개 중 0층은 100만, 1층 약 6만, 2층 약 4천, 3층 약 250, 4층 약 15 … KTX 역처럼 위로 갈수록 드물다.

## 2. 2차원에서 층별 모습과 검색 경로

### 셀 3 · 2차원에서 층별 모습과 검색 경로

In [ ]:
# 셀 3 · 2차원에서 층별 모습과 검색 경로
X2, _ = make_clusters(2000, 2, n_clusters=12, spread=1.0, center_scale=6, seed=3)
h2 = HNSW(M=6, ef_construction=32, seed=1).build(X2)
# 질의: 진입점에서 가장 먼 데이터 점 근처 (멀리 가야 하는 어려운 질의)
far = int(np.argmax(np.linalg.norm(X2 - X2[h2.entry], axis=1)))
qpt = X2[far] + np.array([0.3, 0.3], dtype=np.float32)
trace = []
h2.search(qpt, k=1, ef=8, trace=trace)

layers = [l for l in range(h2.top, -1, -1) if sum(len(L) > l for L in h2.links) > 1][:4]   # 노드가 2개 이상인 층
lim = X2.min(0) - 1, X2.max(0) + 1
fig, axes = plt.subplots(1, len(layers), figsize=(4.2 * len(layers), 4.4))
for ax, lev in zip(axes, layers):
    nodes = [i for i in range(len(X2)) if len(h2.links[i]) > lev]
    for i in nodes:
        for j in h2.links[i][lev]:
            if i < j:
                ax.plot(*X2[[i, j]].T, color="#9aa5a8", lw=0.4 if lev == 0 else 0.9, zorder=1)
    ax.scatter(*X2[nodes].T, s=4 if lev == 0 else 18, color="#3D484A", zorder=2)
    path = [c for l, c in trace if l == lev]
    ax.plot(*X2[path].T, "-o", color="#FF7043", ms=4, lw=2, zorder=3)
    ax.scatter(*qpt, marker="*", s=250, color="#009CA6", zorder=4)
    ax.set_xlim(lim[0][0], lim[1][0]); ax.set_ylim(lim[0][1], lim[1][1])
    ax.set_title(f"{lev}층 · 노드 {len(nodes):,}개 · 방문 {len(path)}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**관찰:** 위층은 노드가 몇 개뿐이라 몇 걸음 만에 질의(★) 쪽으로 멀리 건너간다 (주황 = 방문한 노드).
아래층은 **위층이 찾아 준 출발점**에서 시작하므로 헛걸음이 적다.

## 3. 계층의 효과 — 점이 많아질수록
같은 그래프에서 ① 위층부터 내려오는 HNSW 와 ② 0층에서 바로 시작하는 평면 NSW 를 비교한다.
16차원 군집 데이터, 질의 200개, recall@10 과 **질의당 거리 계산 횟수**를 잰다.

### 셀 4 · 계층의 효과 — HNSW vs 평면 NSW

In [ ]:
# 셀 4 · 계층의 효과 — HNSW vs 평면 NSW
rows = []
for n in [1_000, 4_000, 16_000]:
    X, _ = make_clusters(n + 200, 16, n_clusters=50, seed=5)
    X, Q = X[:n], X[n:]
    truth = brute_knn(X, Q, 10)
    with timer() as t:
        h = HNSW(M=8, ef_construction=48).build(X)
    for flat in [False, True]:
        h.n_dist = 0
        found = [h.search(qq, k=10, ef=32, flat=flat) for qq in Q]
        rows.append({"N": n, "방식": "평면 NSW (0층부터)" if flat else "HNSW (위층부터)",
                     "recall@10": recall(found, truth), "거리계산/질의": h.n_dist / len(Q),
                     "전수비교": n, "빌드(s)": round(t["s"], 1)})
res = pd.DataFrame(rows)
res.round(3)

### 셀 5 · N 이 16배가 될 때 비교 횟수

In [ ]:
# 셀 5 · N 이 16배가 될 때 비교 횟수
fig, ax = plt.subplots(figsize=(7, 4))
for name, g in res.groupby("방식"):
    ax.plot(g["N"], g["거리계산/질의"], "-o", label=name)
ax.plot(res["N"].unique(), res["N"].unique(), "--", color="gray", label="전수 비교 (N)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("N"); ax.set_ylabel("질의당 거리 계산")
ax.legend(); ax.set_title("N 이 16배가 될 때 비교 횟수는?"); plt.show()

**관찰:**
- 전수 비교는 N 에 정비례. HNSW 는 N 이 16배가 돼도 비교 횟수가 **조금만** 는다 (로그에 가까운 증가)
- N 이 작을 때는 평면 NSW 와 차이가 없다. N 이 커지면 평면 NSW 는 **군집 사이를 건너지 못해 recall 이 떨어진다**
  (16,000개에서 0.99 vs 0.81). 위층이 "좋은 출발점"을 만들어 주는 효과다
- 구현이 파이썬이라 빌드는 느리다(pgvector 는 C 로 같은 일을 수십 배 빨리 한다). 여기서는 **횟수**만 본다

## 4. ef_search — recall 과 비용의 교환
같은 인덱스에서 검색 시점의 `ef` 만 바꾼다. 이웃 선택 방식(단순 vs 휴리스틱)도 비교한다.

### 셀 6 · ef_search 와 이웃 선택 방식

In [ ]:
# 셀 6 · ef_search 와 이웃 선택 방식
X, _ = make_clusters(16_200, 16, n_clusters=50, seed=5)
X, Q = X[:16_000], X[16_000:]
truth = brute_knn(X, Q, 10)
curves = {}
for heur in [True, False]:
    h = HNSW(M=8, ef_construction=48, heuristic=heur).build(X)
    pts = []
    for ef in [10, 16, 32, 64, 128]:
        h.n_dist = 0
        found = [h.search(qq, k=10, ef=ef) for qq in Q]
        pts.append((ef, recall(found, truth), h.n_dist / len(Q)))
    curves["휴리스틱 이웃 선택" if heur else "단순: 가장 가까운 M개"] = pts

fig, ax = plt.subplots(figsize=(7, 4))
for name, pts in curves.items():
    ef, r, c = zip(*pts)
    ax.plot(c, r, "-o", label=name)
    for e, x, y in pts:
        ax.annotate(f"ef={e}", (y, x), textcoords="offset points", xytext=(4, -10), fontsize=8)
ax.set_xlabel("질의당 거리 계산"); ax.set_ylabel("recall@10"); ax.legend(); ax.set_title("ef_search ↑ → recall ↑ · 비용 ↑")
plt.show()
pd.DataFrame({k: [f"{r:.3f} / {c:.0f}" for _, r, c in v] for k, v in curves.items()},
             index=[f"ef={e}" for e, _, _ in next(iter(curves.values()))]).rename_axis("recall / 거리계산")

**관찰:**
- `ef` 를 키우면 recall 이 0.9 대 → 0.99 이상으로 오르고 비교 횟수는 늘어난다 (슬라이드 원리 4)
- 이웃을 **다양한 방향으로** 고르는 휴리스틱이 군집 데이터에서 같은 비용 대비 recall 이 높다.
  가장 가까운 M 개만 고르면 이웃이 한 군집 안에 몰려 다른 군집으로 가는 다리가 약해진다
- recall@10 은 "정확한 top-10 중 몇 개를 찾았나" = **ANN recall**. 검색 품질(정답 문서를 찾았나)과는 다른 개념이다 → 30번